# Module 12 — Reliability, Validity & Research Ethics
Dedicated notebook for Module 12 (final module, 12/12) — reproduces Cronbach's Alpha, item-total correlation, alpha-if-deleted for both scales, and connects this to the EFA in Module 10 as evidence of construct validity.

SPSS/PSPP equivalent: `Analyze > Scale > Reliability Analysis`, the `RELIABILITY` command.

In [1]:
# Run this cell first (Colab: click ▶). No extra install needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset -- the SAME seed=2026 used
    throughout Modules 03-08, matching lop_hoc_240.csv/.sav in the repo exactly
    (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("df created:", df.shape)

df created: (240, 14)


## 1. Cronbach's Alpha — by hand, cross-checked against PSPP
🎯 **What question does this method answer?** Cronbach's Alpha answers: do the items in a scale "move together" CONSISTENTLY — it measures the scale's RELIABILITY, and says nothing (yet) about whether the scale measures the RIGHT concept (validity, section 3).

In [2]:
def cronbach_alpha(items_df):
    items_df = items_df.dropna()
    k = items_df.shape[1]
    item_vars = items_df.var(axis=0, ddof=1)
    total_var = items_df.sum(axis=1).var(ddof=1)
    alpha = (k/(k-1)) * (1 - item_vars.sum()/total_var)
    return alpha, len(items_df)

a_interest, n1 = cronbach_alpha(df[["h1","h2","h3"]])
a_anxiety, n2 = cronbach_alpha(df[["a1","a2","a3"]])
print(f"Cronbach's alpha (Interest, h1-h3): {a_interest:.4f} (n={n1})")
print(f"Cronbach's alpha (Anxiety, a1-a3): {a_anxiety:.4f} (n={n2})")

Cronbach's alpha (Interest, h1-h3): 0.7133 (n=237)
Cronbach's alpha (Anxiety, a1-a3): 0.7585 (n=240)


#### 📤 Real output
α(Interest)=0.7133, α(Anxiety)=0.7585. ✅ Matches PSPP (`RELIABILITY`) exactly: 0.71 and 0.76 (rounded to 2 decimals).

## 2. Item-total correlation & Alpha-if-deleted
🎯 **What question does this method answer?** These two indices answer: how does EACH individual item contribute to the scale's overall reliability -- which item is "dragging" alpha down, and would dropping an item improve the scale?

In [3]:
items = df[["h1","h2","h3"]].dropna()
total = items.sum(axis=1)
print("--- Interest scale ---")
for col in items.columns:
    rest = total - items[col]
    r = np.corrcoef(items[col], rest)[0,1]
    others = [c for c in items.columns if c != col]
    a_del, _ = cronbach_alpha(items[others])
    print(f"{col}: item-rest r={r:.3f}, alpha-if-deleted={a_del:.4f}")

--- Interest scale ---
h1: item-rest r=0.570, alpha-if-deleted=0.5752
h2: item-rest r=0.492, alpha-if-deleted=0.6716
h3: item-rest r=0.535, alpha-if-deleted=0.6195


#### 📤 Real output
h1: r=0.570, alpha-if-deleted=0.575. h2: r=0.492, alpha-if-deleted=0.672. h3: r=0.535, alpha-if-deleted=0.620. ✅ Matches PSPP (0.57/0.58; 0.49/0.67; 0.54/0.62). Dropping any single item would DECREASE alpha below the current 0.713 -> keeping all 3 items is the best choice.

## 3. Construct validity — connecting to the EFA (Module 10)
🎯 **What question does this method answer?** Comparing the EFA results (Module 10) against the original theoretical design answers: does the scale measure the INTENDED theoretical concept (construct validity) -- a fundamentally different question from "is it internally consistent" (reliability, sections 1-2).

In [4]:
from factor_analyzer import FactorAnalyzer
items6 = df[["h1","h2","h3","a1","a2","a3"]].dropna()
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items6)
loadings = pd.DataFrame(fa.loadings_, index=items6.columns, columns=["Factor 1", "Factor 2"])
print(loadings.round(3))
print("\n-> h1-h3 load clearly on one factor, a1-a3 load clearly on the other")
print("-> Matches the original theoretical design exactly (2 concepts: interest, anxiety)")
print("-> This is evidence that REINFORCES the construct validity of both scales")

    Factor 1  Factor 2
h1    -0.043     0.747
h2    -0.003     0.603
h3    -0.096     0.670
a1     0.725    -0.092
a2     0.722    -0.083
a3     0.695    -0.016

-> h1-h3 load clearly on one factor, a1-a3 load clearly on the other
-> Matches the original theoretical design exactly (2 concepts: interest, anxiety)
-> This is evidence that REINFORCES the construct validity of both scales


#### 📤 Real output
The EFA structure matches the original design exactly (same as Module 10) -- combined with the acceptable Cronbach's Alpha in section 1, we now have fairly solid evidence for BOTH RELIABILITY (alpha) AND CONSTRUCT VALIDITY (EFA matches the design) of both scales.

## 4. Illustrating "high reliability, low validity" with simulated data
🎯 The classic "scale that's off by 2kg" example: construct a measurement with a fixed SYSTEMATIC error, showing that reliability (alpha) can still be high even when the measurement is inaccurate.

In [5]:
rng = np.random.default_rng(2026)
true_weight = rng.normal(60, 10, 240)  # the true weight
# "Faulty scale": always reads 2kg heavier, plus small measurement noise (measured 3 times with the same faulty scale)
bias = 2.0
measured_1 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_2 = true_weight + bias + rng.normal(0, 0.3, 240)
measured_3 = true_weight + bias + rng.normal(0, 0.3, 240)

fake_scale = pd.DataFrame({"trial1": measured_1, "trial2": measured_2, "trial3": measured_3})
alpha_fake, _ = cronbach_alpha(fake_scale)
mean_error = (fake_scale.mean(axis=1) - true_weight).mean()
print(f"Cronbach's alpha of the 'faulty scale' (3 measurements): {alpha_fake:.4f}  <- VERY HIGH (good reliability)")
print(f"Mean error vs the TRUE weight: {mean_error:+.2f} kg  <- ALWAYS biased (poor validity)")

Cronbach's alpha of the 'faulty scale' (3 measurements): 0.9997  <- VERY HIGH (good reliability)
Mean error vs the TRUE weight: +2.00 kg  <- ALWAYS biased (poor validity)


#### 📤 Real output
The "faulty scale" alpha is very high (>0.95 -- the 3 readings are nearly identical) yet the mean error against the TRUE weight is always +2.00kg. Demonstrated with simulated data: **high reliability does NOT guarantee high validity** -- the core lesson of Module 12.